# 06 — Rewiring vs heterochrony, and what drives cross-species similarity

**Claims under test**
1. Positional genes are **redeployed**, not merely retimed: Δpeak median 0, ~33% displaced >=5
   deciles, and this **replicates on a second coordinate** (rho=0.732 between axes).
2. Decidual correspondence is carried substantially by housekeeping genes (10.6%) while fibroblast
   correspondence is not (0.7%).

**Correction carried in the code:** the Δpeak tails are **asymmetric** (+9: 36 genes, −9: 17), not
symmetric — so the honest reading is predominantly rewiring **with a minor human-later heterochronic
component** (Wilcoxon p=3.6e-7).

In [1]:
# --- paths & helpers (edit ROOT if you move things) ---------------------------
import os, json, glob, pathlib, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

X    = pathlib.Path("/n/lab_storage/eddy_lab/users/nhilgert/xmens_final")
REV  = pathlib.Path("/n/lab_storage/eddy_lab/users/nhilgert/decidualization/revisions")
NET  = pathlib.Path("/n/netscratch/eddy_lab/Everyone/nhilgert")
GP   = REV/"crossspecies_radial_gp"
DELIV= X/"data/deliverables"
ISO  = X/"results/isodepth"
MIS  = X/"results/mi_stratified"
DATA = pathlib.Path("data")     # staged artifacts, see README
ORTH = pathlib.Path("/n/lab_storage/eddy_lab/users/nhilgert/human_ivd_repro/orthogroups/ensembl_human_mouse_components.tsv")

HS_DELIV = DELIV/"2026_08_28_Cevrim_Human_Only_Fib_Dec_Cells.h5ad"
MM_DELIV = DELIV/"2026_08_28_Cevrim_XMens_Only_Fib_Dec_Cells.h5ad"   # NOTE: "XMens", not "Mouse"

_checks = []
def verify(label, got, expected, tol=0.02, note=""):
    """Recompute vs recorded. Prints PASS/MISMATCH and accumulates a summary."""
    try:
        ok = (abs(float(got) - float(expected)) <= tol * max(1.0, abs(float(expected)))
              if expected is not None else None)
    except (TypeError, ValueError):
        ok = (got == expected)
    tag = "PASS " if ok else ("MISMATCH" if ok is False else "n/a  ")
    _checks.append((label, got, expected, tag))
    print(f"[{tag}] {label}: recomputed={got}  recorded={expected}" + (f"   ({note})" if note else ""))
    return ok

def check_summary():
    d = pd.DataFrame(_checks, columns=["claim","recomputed","recorded","status"])
    n_bad = int((d.status.str.strip()=="MISMATCH").sum())
    print(f"\n{len(d)} checks | {n_bad} MISMATCH")
    return d

def ortholog_1to1():
    o = pd.read_csv(ORTH, sep="\t")
    o = o[o.homology_type=="ortholog_one2one"]
    piv = o.pivot_table(index="component_id", columns="species", values="gene_symbol",
                        aggfunc=lambda s: s.iloc[0] if len(s)==1 else np.nan).dropna()
    return piv[["human","mouse"]].drop_duplicates()

print("data dir:", DATA.resolve(), "| files:", len(list(DATA.glob("*"))) if DATA.exists() else "MISSING")
print("env ok |", "deliverables:", HS_DELIV.exists(), MM_DELIV.exists(), "| isodepth dir:", ISO.exists())


data dir: /n/lab_storage/eddy_lab/users/nhilgert/xmens_final/notebooks/verify/data | files: 25
env ok | deliverables: True True | isodepth dir: True


## 6.1 Δpeak on two axes

In [2]:
p_ = DATA/"dpeak_two_axes.json"
if not p_.exists(): p_ = None
if p_:
    d = json.load(open(p_))
    for ax in ("isodepth_axis","pseudotime_axis"):
        if ax in d:
            v = d[ax]
            print(f"{ax:16s} n={v['n']} median={v['median']} mean={v['mean']} sd={v['sd']} "
                  f"|<=1|={v['frac_le1']} |>=5|={v['frac_ge5']} wilcoxon_p={v['wilcoxon_p']:.2e}")
    if "agreement" in d:
        print("\nagreement:", json.dumps(d["agreement"], indent=1))
        verify("the two axes agree on displacement", d["agreement"]["spearman_dpeak_iso_vs_pt"], 0.732)
    verify("isodepth median dpeak", d["isodepth_axis"]["median"], 0.0, tol=0)
    verify("frac displaced >=5 (isodepth)", d["isodepth_axis"]["frac_ge1" ] if False else d["isodepth_axis"]["frac_ge5"], 0.331)
    verify("mean is POSITIVE (human-later skew)", bool(d["isodepth_axis"]["mean"] > 0), True,
           note="minor heterochronic component; tails are NOT symmetric")

isodepth_axis    n=478 median=0.0 mean=1.056 sd=4.363 |<=1|=0.416 |>=5|=0.331 wilcoxon_p=3.59e-07
pseudotime_axis  n=478 median=1.0 mean=1.345 sd=3.924 |<=1|=0.389 |>=5|=0.31 wilcoxon_p=1.71e-13

agreement: {
 "spearman_dpeak_iso_vs_pt": 0.732,
 "frac_same_sign": 0.697,
 "frac_both_displaced": 0.226,
 "frac_both_matched": 0.291
}
[PASS ] the two axes agree on displacement: recomputed=0.732  recorded=0.732
[PASS ] isodepth median dpeak: recomputed=0.0  recorded=0.0
[PASS ] frac displaced >=5 (isodepth): recomputed=0.331  recorded=0.331
[PASS ] mean is POSITIVE (human-later skew): recomputed=True  recorded=True   (minor heterochronic component; tails are NOT symmetric)


## 6.2 The tail asymmetry — recompute directly

Recorded: −9 -> 17 genes, +9 -> 36 genes, 0 -> 105. I once reported "36 and 36"; this cell makes that uncheckable-by-eye claim explicit.

In [3]:
p_ = DATA/"conservation_ranking_all_genes.tsv"
if not p_.exists(): p_ = None
if p_:
    T = pd.read_csv(p_, sep="\t")
    pos = T[(T.pct_human>=0.90) & (T.pct_mouse>=0.90)] if "pct_human" in T.columns else T
    if {"peak_dec_human","peak_dec_mouse"} <= set(T.columns):
        dp = (pos.peak_dec_human - pos.peak_dec_mouse)
        vc = dp.value_counts().sort_index()
        print("dpeak distribution (positional in both):")
        print(vc.to_string())
        verify("n positional in both", len(pos), 478, tol=0.02)
        verify("dpeak = -9 count", int(vc.get(-9,0)), 17, tol=0)
        verify("dpeak = +9 count", int(vc.get(9,0)), 36, tol=0)
        verify("tails are ASYMMETRIC toward human-later", bool(vc.get(9,0) > vc.get(-9,0)), True)
else:
    print("conservation_ranking_all_genes.tsv not found")

dpeak distribution (positional in both):
-9     17
-8      6
-7     10
-6      6
-5      5
-4      8
-3     20
-2     34
-1     36
 0    105
 1     58
 2     22
 3     19
 4     18
 5     30
 6     19
 7     18
 8     11
 9     36
[PASS ] n positional in both: recomputed=478  recorded=478
[PASS ] dpeak = -9 count: recomputed=17  recorded=17
[PASS ] dpeak = +9 count: recomputed=36  recorded=36
[PASS ] tails are ASYMMETRIC toward human-later: recomputed=True  recorded=True


## 6.3 SAMap gene-pair drivers — the housekeeping contrast

Recorded (12 runs, after dropping promiscuous many-to-one pairs): dec↔dec **10.6%** housekeeping vs
fib↔fib **0.7%**, and both compartments' drivers sit at the ~92nd percentile of isodepth MI.

In [4]:
p_ = DATA/"genepairs_v3_all.csv"
if not p_.exists(): p_ = None
if p_:
    D = pd.read_csv(p_)
    print("runs:", D.run.nunique(), "| pairs:", len(D))
    for tag, g in D.groupby("tag"):
        cg = g[(~g.housekeeping) & (~g.promiscuous)]
        print(f"  {tag}: n={len(g)} HK={100*g.housekeeping.mean():.1f}% promisc={100*g.promiscuous.mean():.1f}%"
              f" | clean n={len(cg)} medianMIpct h={cg.mi_pct_human.median():.3f} m={cg.mi_pct_mouse.median():.3f}")
    dec = D[D.tag=="dec_x_dec"].housekeeping.mean(); fib = D[D.tag=="fib_x_fib"].housekeeping.mean()
    verify("dec HK fraction", round(float(dec),3), 0.106)
    verify("fib HK fraction", round(float(fib),3), 0.007)
    verify("dec is >10x more housekeeping-carried than fib", bool(dec > 10*fib), True)
    print("\nper-run dec HK (no monotone trend with mapping score - do not read one):")
    print(D[D.tag=="dec_x_dec"].groupby("run").housekeeping.mean().round(3).to_string())
else:
    print("genepairs_v3_all.csv not found")

runs: 12 | pairs: 13895
  dec_x_dec: n=9828 HK=10.6% promisc=29.1% | clean n=6158 medianMIpct h=0.919 m=0.904
  fib_x_fib: n=4067 HK=0.7% promisc=22.4% | clean n=3156 medianMIpct h=0.932 m=0.931
[PASS ] dec HK fraction: recomputed=0.106  recorded=0.106
[PASS ] fib HK fraction: recomputed=0.007  recorded=0.007
[PASS ] dec is >10x more housekeeping-carried than fib: recomputed=True  recorded=True

per-run dec HK (no monotone trend with mapping score - do not read one):
run
menstrual_x_1dpi_early    0.160
menstrual_x_1dpi_late     0.090
menstrual_x_2dpi          0.039
menstrual_x_3dpi          0.057
menstrual_x_4dpi          0.139
menstrual_x_uninduced     0.083
secretory_x_1dpi_early    0.167
secretory_x_1dpi_late     0.101
secretory_x_2dpi          0.045
secretory_x_3dpi          0.070
secretory_x_4dpi          0.206
secretory_x_uninduced     0.102


## 6.4 Verified cross-method convergence

Genes named by **both** the MI-gain analysis and the SAMap driver lists — two methods sharing no machinery.

In [5]:
CONVERGENT = {
 "decidual pole": ["RUNX1/Runx1", "CEBPB/Cebpb", "GJA1/Gja1", "CD63/Cd63", "NAMPT/Nampt"],
 "fibroblast pole": ["MEIS2/Meis2", "EBF1/Ebf1", "NFIA/Nfia", "RBMS3/Rbms3", "RORA/Rora"],
}
for k_, v in CONVERGENT.items():
    print(f"{k_}: {', '.join(v)}")
print("\nNOT convergent (asserted once in error): WNT5A - it is a LOSS gene (gain z = -1.91),")
print("and its SAMap pair is hs_WNT5A;mm_Wnt4, a different mouse gene than the 1:1 Wnt5a used for MI.")

decidual pole: RUNX1/Runx1, CEBPB/Cebpb, GJA1/Gja1, CD63/Cd63, NAMPT/Nampt
fibroblast pole: MEIS2/Meis2, EBF1/Ebf1, NFIA/Nfia, RBMS3/Rbms3, RORA/Rora

NOT convergent (asserted once in error): WNT5A - it is a LOSS gene (gain z = -1.91),
and its SAMap pair is hs_WNT5A;mm_Wnt4, a different mouse gene than the 1:1 Wnt5a used for MI.


## Summary

In [6]:
check_summary()


11 checks | 0 MISMATCH


,claim,recomputed,recorded,status
0,the two axes agree on displacement,0.732,0.732,PASS
1,isodepth median dpeak,0.0,0.0,PASS
2,frac displaced >=5 (isodepth),0.331,0.331,PASS
3,mean is POSITIVE (human-later skew),True,True,PASS
4,n positional in both,478,478,PASS
5,dpeak = -9 count,17,17,PASS
6,dpeak = +9 count,36,36,PASS
7,tails are ASYMMETRIC toward human-later,True,True,PASS
8,dec HK fraction,0.106,0.106,PASS
9,fib HK fraction,0.007,0.007,PASS
